In [2]:
import re
from datetime import datetime

def group_key(filename, time_bucket_sec=30, frame_bucket_size=20):
    """
    Groups image filenames by timestamp, frame number, or trailing-number pattern.
    """
    name = os.path.splitext(os.path.basename(filename))[0]

    # Timestamp format:
    # vlcsnap-2025-02-18-17h01m16s802
    # vlcsnap_2025-03-16-15h31m42s236
    timestamp_pattern = (
        r"^vlcsnap[-_](\d{4})-(\d{2})-(\d{2})-"
        r"(\d{2})h(\d{2})m(\d{2})s\d*$"
    )
    match = re.match(timestamp_pattern, name, re.IGNORECASE)

    if match:
        year, month, day, hour, minute, second = map(int, match.groups())
        try:
            dt = datetime(year, month, day, hour, minute, second)
            bucket = int(dt.timestamp()) // time_bucket_sec
            return f"ts_{year:04d}{month:02d}{day:02d}_{bucket}"
        except ValueError:
            pass

    # Frame format: vlcsnap-00001 or vlcsnap-00097-MSI
    match = re.match(r"^([A-Za-z]+)-(\d+)", name)
    if match:
        prefix, number = match.groups()
        bucket = int(number) // frame_bucket_size
        return f"{prefix}_frame_{bucket}"

    # Fallback: remove trailing numbers
    return re.sub(r"[_-]?\d+$", "", name)

## 🧪 ทดสอบด้วยโค้ดเช็คเดิม (ปรับให้ใช้ฟังก์ชันใหม่)

#```python
import os, glob
from collections import defaultdict

SRC_DIR = os.path.join(os.getcwd(), "data")
img_paths = sorted(glob.glob(os.path.join(SRC_DIR, 'images', '*.*')))

groups = defaultdict(list)
for p in img_paths:
    fname = os.path.basename(p)
    groups[group_key(fname)].append(fname)

sizes = [len(v) for v in groups.values()]
print(f"📊 จำนวนกลุ่มทั้งหมด: {len(groups)}")
print(f"   เฉลี่ย: {sum(sizes)/len(sizes):.2f} ไฟล์/กลุ่ม")
print(f"   มากสุด: {max(sizes)} ไฟล์")
print(f"   น้อยสุด: {min(sizes)} ไฟล์")

# แสดงตัวอย่าง 5 กลุ่มแรก
for k, v in list(groups.items())[:5]:
    print(f"\nKey: '{k}' ({len(v)} ไฟล์)")
    for f in v[:3]:
        print(f"  - {f}")

📊 จำนวนกลุ่มทั้งหมด: 295
   เฉลี่ย: 6.81 ไฟล์/กลุ่ม
   มากสุด: 60 ไฟล์
   น้อยสุด: 1 ไฟล์

Key: '20250216' (3 ไฟล์)
  - 20250216_164325.jpg
  - 20250216_164521.jpg
  - 20250216_164541.jpg

Key: '20250219' (60 ไฟล์)
  - 20250219_164649.jpg
  - 20250219_164714.jpg
  - 20250219_164738.jpg

Key: '20250223' (39 ไฟล์)
  - 20250223_104730.jpg
  - 20250223_104743.jpg
  - 20250223_104750.jpg

Key: 'vlcsnap_frame_0' (19 ไฟล์)
  - vlcsnap-00001.jpg
  - vlcsnap-00002.jpg
  - vlcsnap-00003.jpg

Key: 'vlcsnap_frame_1' (20 ไฟล์)
  - vlcsnap-00020.jpg
  - vlcsnap-00021.jpg
  - vlcsnap-00022.jpg


In [3]:
import os, re, glob
from datetime import datetime
from collections import defaultdict

def extract_timestamp(filename):
    name = os.path.splitext(os.path.basename(filename))[0]
    patterns = [
        r"^vlcsnap[_-](\d{4})-(\d{2})-(\d{2})-(\d{2})h(\d{2})m(\d{2})s\d*$",
        r"^(\d{8})_(\d{6})$",
    ]
    for pattern in patterns:
        m = re.match(pattern, name, re.IGNORECASE)
        if m:
            if len(m.groups()) == 7:
                y, mo, d, h, mi, s, _ = m.groups()
                return datetime(int(y), int(mo), int(d), int(h), int(mi), int(s))
            return datetime.strptime("".join(m.groups()), "%Y%m%d%H%M%S")
    return None

def extract_frame_number(filename):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.match(r"^([A-Za-z]+)-(\d+)(?:-[A-Za-z0-9]+)?$", name)
    if m:
        return m.group(1), int(m.group(2))
    return None, None

def build_groups(img_paths, time_gap_sec=90, frame_gap=5, max_group_size=25):
    with_time, with_frame, others = [], [], []
    for p in img_paths:
        fname = os.path.basename(p)
        ts = extract_timestamp(fname)
        if ts:
            with_time.append((ts, p))
            continue
        prefix, num = extract_frame_number(fname)
        if prefix is not None:
            with_frame.append((prefix, num, p))
            continue
        others.append(p)

    raw_groups = defaultdict(list)
    gid = 0

    with_time.sort(key=lambda x: x[0])
    prev_ts = None
    for ts, p in with_time:
        if prev_ts is None or (ts - prev_ts).total_seconds() > time_gap_sec:
            gid += 1
        raw_groups[f"time_g{gid}"].append(p)
        prev_ts = ts

    with_frame.sort(key=lambda x: (x[0], x[1]))
    prev_prefix, prev_num = None, None
    for prefix, num, p in with_frame:
        if prefix != prev_prefix or (num - prev_num) > frame_gap:
            gid += 1
        raw_groups[f"frame_g{gid}"].append(p)
        prev_prefix, prev_num = prefix, num

    for p in others:
        gid += 1
        raw_groups[f"other_g{gid}"].append(p)

    final_map = {}
    for key, files in raw_groups.items():
        if len(files) <= max_group_size:
            for f in files:
                final_map[f] = key
        else:
            for i, f in enumerate(files):
                chunk_id = i // max_group_size
                final_map[f] = f"{key}_c{chunk_id}"

    return final_map


# ==================== ทดสอบ ====================
SRC_DIR = os.path.join(os.getcwd(), "data")
img_paths = sorted(glob.glob(os.path.join(SRC_DIR, 'images', '*.*')))

group_map = build_groups(img_paths, time_gap_sec=90, frame_gap=5, max_group_size=25)
groups = defaultdict(list)
for p, k in group_map.items():
    groups[k].append(os.path.basename(p))

sizes = [len(v) for v in groups.values()]
print(f"📊 จำนวนกลุ่มทั้งหมด: {len(groups)}")
print(f"   เฉลี่ย: {sum(sizes)/len(sizes):.2f} ไฟล์/กลุ่ม")
print(f"   มากสุด: {max(sizes)} ไฟล์")
print(f"   น้อยสุด: {min(sizes)} ไฟล์")

for k, v in sorted(groups.items(), key=lambda x: -len(x[1]))[:5]:
    print(f"\nKey: '{k}' ({len(v)} ไฟล์)")
    for f in v[:3]:
        print(f"  - {f}")

📊 จำนวนกลุ่มทั้งหมด: 99
   เฉลี่ย: 20.29 ไฟล์/กลุ่ม
   มากสุด: 25 ไฟล์
   น้อยสุด: 1 ไฟล์

Key: 'time_g3_c0' (25 ไฟล์)
  - vlcsnap-2025-02-18-17h01m16s802.jpg
  - vlcsnap-2025-02-18-17h01m19s628.jpg
  - vlcsnap-2025-02-18-17h01m26s641.jpg

Key: 'time_g3_c1' (25 ไฟล์)
  - vlcsnap-2025-02-18-17h02m44s073.jpg
  - vlcsnap-2025-02-18-17h02m46s063.jpg
  - vlcsnap-2025-02-18-17h02m47s744.jpg

Key: 'time_g3_c2' (25 ไฟล์)
  - vlcsnap-2025-02-18-17h04m14s457.jpg
  - vlcsnap-2025-02-18-17h04m16s176.jpg
  - vlcsnap-2025-02-18-17h04m19s949.jpg

Key: 'time_g4_c0' (25 ไฟล์)
  - vlcsnap-2025-02-18-17h10m25s907.jpg
  - vlcsnap-2025-02-18-17h10m28s875.jpg
  - vlcsnap-2025-02-18-17h10m30s786.jpg

Key: 'time_g5_c0' (25 ไฟล์)
  - vlcsnap-2025-02-18-18h30m17s408.jpg
  - vlcsnap-2025-02-18-18h30m20s293.jpg
  - vlcsnap-2025-02-18-18h30m22s081.jpg


In [4]:
import sys
import subprocess

try:
    import torch
except ModuleNotFoundError:
    print("Installing PyTorch into the active Jupyter kernel...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch",
        "torchvision",
        "torchaudio",
    ])
    import torch

print(f"PyTorch {torch.__version__} loaded")
print(f"CUDA available: {torch.cuda.is_available()}")

Installing PyTorch into the active Jupyter kernel...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 975.8 kB/s  0:13:25 eta 0:00:010:00:21
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 1.4 MB/s  0:06:48 eta 0:00:010:00:12m
Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl (170.1 MB)
Using cached nvidia_nccl_cu13-2.30.7-py3-none-manylinux_2_18_x86_64.whl (216.0 MB)
Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (60.4 MB)
Using cached cuda_bindings-13.4.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (7.0 MB)
Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl (423.1 MB)
Using cached nvidia_cuda_cupti-13.0.85-py3-none-manylinux_2_25_x86_64.whl (10.7 MB)
Using cached nvidia_cuda_nvrtc-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl (90.2 MB)
Using cached nvidia_cuda_runtime-13.0.96-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (2.2 M

In [10]:
"""
================================================================
FULL PIPELINE: Road Damage Detection (YOLO11)
รวม: Smart Group-Aware Split + Data Integrity Check
     + Tuned Hyperparameters + Full Augmentation
     + Train + Evaluate (val/test) + Export ONNX
================================================================
"""
import os, re, gc, glob, random, yaml, shutil
import cv2
from datetime import datetime
from collections import defaultdict
import numpy as np
import torch
from ultralytics import YOLO

cv2.setNumThreads(0)

## Check GPU

def get_device():
    """ตรวจสอบว่ามี GPU (CUDA) ใช้งานได้หรือไม่ ถ้าไม่มีให้ใช้ CPU แทน"""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ พบ GPU: {gpu_name} -> ใช้ device='0'")
        return 0
    else:
        print("⚠️ ไม่พบ GPU (CUDA) -> จะใช้ CPU แทน (การเทรนจะช้ากว่ามาก)")
        return "cpu"

DEVICE  = get_device()

    # เปิด cudnn optimization เฉพาะตอนมี GPU เท่านั้น
if DEVICE != "cpu":
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ==================== 0) CONFIG & REPRODUCIBILITY ====================
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

BASE_DIR   = os.getcwd()
SRC_DIR    = os.path.join(BASE_DIR, "data")           # ข้อมูลดิบต้นฉบับ
WORK_DIR   = os.path.join(BASE_DIR, "dataset_clean")  # ข้อมูลหลัง split
PROJECT    = "road-damage-final"
RUN_NAME   = "merged_best"
CLASS_NAMES = ['pothole', 'crack', 'manhole']

HAS_GPU = torch.cuda.is_available()
BATCH   = 32 if DEVICE != "cpu" else 4 
WORKERS = 16

EPOCHS   = 200
IMGSZ    = 896
PATIENCE = 50

# Hyperparameter ที่ผ่าน Optuna tuning
OPTIMIZER    = "RAdam"
LR0          = 0.007023087386876883
LRF          = 0.02559244954708425
WEIGHT_DECAY = 0.005034138135400871
MOMENTUM     = 0.09079056013311568
DROPOUT      = 0.11321865578015432

# Grouping config
TIME_GAP_SEC   = 90   # ห่างเกินนี้ = คนละฉาก (สำหรับไฟล์ timestamp)
FRAME_GAP      = 5    # เลขรันห่างเกินนี้ = คนละฉาก (สำหรับไฟล์เลขรัน)
MAX_GROUP_SIZE = 25   # ล็อกไม่ให้กลุ่มไหนใหญ่เกินนี้

print(f"🖥️  Device: {'GPU' if HAS_GPU else 'CPU'} | Batch: {BATCH}")

# ==================== 1) SMART GROUP KEY FUNCTIONS ====================
def extract_timestamp(filename):
    """ดึงเวลาจากชื่อไฟล์ รองรับหลายรูปแบบ"""
    name = os.path.splitext(os.path.basename(filename))[0]
    patterns = [
        r"^vlcsnap[_-](\d{4})-(\d{2})-(\d{2})-(\d{2})h(\d{2})m(\d{2})s\d*$",
        r"^(\d{8})_(\d{6})$",
    ]
    for pattern in patterns:
        m = re.match(pattern, name, re.IGNORECASE)
        if m:
            if len(m.groups()) == 7:
                y, mo, d, h, mi, s, _ = m.groups()
                return datetime(int(y), int(mo), int(d), int(h), int(mi), int(s))
            return datetime.strptime("".join(m.groups()), "%Y%m%d%H%M%S")
    return None

def extract_frame_number(filename):
    """ดึง prefix + เลขรัน เช่น vlcsnap-00001 หรือ vlcsnap-00097-MSI"""
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.match(r"^([A-Za-z]+)-(\d+)(?:-[A-Za-z0-9]+)?$", name)
    if m:
        return m.group(1), int(m.group(2))
    return None, None

def build_groups(img_paths, time_gap_sec=90, frame_gap=5, max_group_size=25):
    """Gap-based clustering + Max Size Cap"""
    with_time, with_frame, others = [], [], []
    for p in img_paths:
        fname = os.path.basename(p)
        ts = extract_timestamp(fname)
        if ts:
            with_time.append((ts, p)); continue
        prefix, num = extract_frame_number(fname)
        if prefix is not None:
            with_frame.append((prefix, num, p)); continue
        others.append(p)

    raw_groups = defaultdict(list)
    gid = 0

    with_time.sort(key=lambda x: x[0])
    prev_ts = None
    for ts, p in with_time:
        if prev_ts is None or (ts - prev_ts).total_seconds() > time_gap_sec:
            gid += 1
        raw_groups[f"time_g{gid}"].append(p)
        prev_ts = ts

    with_frame.sort(key=lambda x: (x[0], x[1]))
    prev_prefix, prev_num = None, None
    for prefix, num, p in with_frame:
        if prefix != prev_prefix or (num - prev_num) > frame_gap:
            gid += 1
        raw_groups[f"frame_g{gid}"].append(p)
        prev_prefix, prev_num = prefix, num

    for p in others:
        gid += 1
        raw_groups[f"other_g{gid}"].append(p)

    final_map = {}
    for key, files in raw_groups.items():
        if len(files) <= max_group_size:
            for f in files:
                final_map[f] = key
        else:
            for i, f in enumerate(files):
                chunk_id = i // max_group_size
                final_map[f] = f"{key}_c{chunk_id}"
    return final_map

# ==================== 2) BUILD GROUPS & CHECK STATS ====================
img_paths = sorted(glob.glob(os.path.join(SRC_DIR, 'images', '*.*')))
print(f"\n📂 พบไฟล์ภาพทั้งหมด: {len(img_paths)} ไฟล์")

group_map = build_groups(img_paths, TIME_GAP_SEC, FRAME_GAP, MAX_GROUP_SIZE)
groups = defaultdict(list)
for p, k in group_map.items():
    groups[k].append(p)

sizes = [len(v) for v in groups.values()]
print(f"📊 จำนวนกลุ่มทั้งหมด: {len(groups)}")
print(f"   เฉลี่ย: {sum(sizes)/len(sizes):.2f} ไฟล์/กลุ่ม | มากสุด: {max(sizes)} | น้อยสุด: {min(sizes)}")

# ==================== 3) GROUP-AWARE SPLIT (75/15/10) ====================
group_keys = list(groups.keys())
random.shuffle(group_keys)

n = len(group_keys)
n_train = int(n * 0.75)
n_val   = int(n * 0.15)
train_keys = group_keys[:n_train]
val_keys   = group_keys[n_train:n_train + n_val]
test_keys  = group_keys[n_train + n_val:]

split_map = {}
for k in train_keys:
    for p in groups[k]: split_map[p] = 'train'
for k in val_keys:
    for p in groups[k]: split_map[p] = 'val'
for k in test_keys:
    for p in groups[k]: split_map[p] = 'test'

def label_path(img_p):
    base = os.path.splitext(os.path.basename(img_p))[0]
    return os.path.join(SRC_DIR, 'labels', base + '.txt')

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(WORK_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(WORK_DIR, 'labels', split), exist_ok=True)

missing_label, copied = 0, 0
for img_p, split in split_map.items():
    lbl_p = label_path(img_p)
    if not os.path.exists(lbl_p):
        missing_label += 1
        continue
    shutil.copy2(img_p, os.path.join(WORK_DIR, 'images', split, os.path.basename(img_p)))
    shutil.copy2(lbl_p, os.path.join(WORK_DIR, 'labels', split, os.path.basename(lbl_p)))
    copied += 1

print(f"\n✅ Split เสร็จ | train={len(train_keys)} groups, val={len(val_keys)} groups, test={len(test_keys)} groups")
print(f"✅ ไฟล์ที่คัดลอกสำเร็จ: {copied} | ⚠️ ภาพที่ไม่มี label: {missing_label}")

# ==================== 4) DATA INTEGRITY CHECK ====================
print("\n🔍 ตรวจสอบคุณภาพข้อมูล:")
for split in ['train', 'val', 'test']:
    lbl_files = glob.glob(os.path.join(WORK_DIR, 'labels', split, '*.txt'))
    img_files = glob.glob(os.path.join(WORK_DIR, 'images', split, '*.*'))
    bad_class = 0
    for lf in lbl_files:
        with open(lf) as f:
            for line in f:
                if not line.strip():
                    continue
                cid = int(line.split()[0])
                if cid >= len(CLASS_NAMES):
                    bad_class += 1
    print(f"  [{split}] images={len(img_files)} | labels={len(lbl_files)} | class id ผิดพลาด={bad_class}")

# ==================== 5) สร้าง data.yaml ====================
yaml_data = {
    'path': WORK_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}
yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)
print(f"\n✅ สร้างไฟล์ {yaml_path} สำเร็จ")

# ==================== 6) โหลดโมเดล (พร้อม Offline Fallback) ====================
weights_path = os.path.join(BASE_DIR, "weights", "yolo11m.pt")
if os.path.exists(weights_path):
    model = YOLO(weights_path)
    print("✅ โหลด yolo11m.pt จากเครื่อง (offline)")
else:
    try:
        model = YOLO('yolo11m.pt')
        print("✅ โหลด yolo11m.pt จากอินเทอร์เน็ต")
    except Exception:
        model = YOLO('yolo11n.yaml')
        print("⚠️ โหลดออนไลน์ไม่สำเร็จ ใช้ yolo11n.yaml (train จากศูนย์) แทน")

# ==================== 7) TRAIN ====================
print("\n🚀 เริ่มเทรนโมเดล...")
# Convert polygon labels to YOLO detection bounding boxes
for split_name in ("train", "val", "test"):
    labels_dir = os.path.join(WORK_DIR, "labels", split_name)

    for label_file in glob.glob(os.path.join(labels_dir, "*.txt")):
        converted = []

        with open(label_file, "r") as f:
            for line in f:
                values = line.strip().split()
                if not values:
                    continue

                class_id = int(values[0])
                coords = list(map(float, values[1:]))

                if len(coords) == 4:
                    # Already in detection format
                    converted.append(
                        f"{class_id} " + " ".join(f"{x:.6f}" for x in coords)
                    )
                elif len(coords) >= 6 and len(coords) % 2 == 0:
                    # Polygon -> bounding box
                    xs = coords[0::2]
                    ys = coords[1::2]

                    xmin, xmax = min(xs), max(xs)
                    ymin, ymax = min(ys), max(ys)

                    xc = (xmin + xmax) / 2
                    yc = (ymin + ymax) / 2
                    w = xmax - xmin
                    h = ymax - ymin

                    converted.append(ago
                        f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
                    )
                else:
                    raise ValueError(
                        f"รูปแบบ label ไม่ถูกต้อง: {label_file}\n{line}"
                    )

        with open(label_file, "w") as f:
            f.write("\n".join(converted) + ("\n" if converted else ""))

print("✅ แปลง polygon labels เป็น bounding boxes แล้ว")

# ใช้ workers=0 ใน Jupyter เพื่อป้องกัน BrokenPipeError
# ใช้ workers=0 ใน Jupyter เพื่อป้องกัน BrokenPipeError
results = model.train(
    data=yaml_path,
    project=PROJECT,
    name=RUN_NAME,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    amp=HAS_GPU,
    deterministic=True,
    seed=SEED,
    workers=WORKERS,
    cache=False,

    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    weight_decay=WEIGHT_DECAY,
    momentum=MOMENTUM,
    dropout=DROPOUT,

    box=9.0,
    cls=0.6,
    dfl=1.6,

    mosaic=1.0,
    close_mosaic=20,
    mixup=0.1,
    copy_paste=0.3,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

torch.cuda.empty_cache()
gc.collect()

# ==================== 8) EVALUATE (val + test พร้อม TTA) ====================
best_pt = os.path.join(results.save_dir, 'weights', 'best.pt')
best = YOLO(best_pt)

print("\n📊 ผลบน VAL SET:")
val_metrics = best.val(data=yaml_path, split='val', conf=0.001, iou=0.6, imgsz=IMGSZ)

print("\n📊 ผลบน TEST SET (พร้อม Test-Time Augmentation):")
test_metrics = best.val(data=yaml_path, split='test', conf=0.001, iou=0.6, imgsz=IMGSZ, augment=True)

print(f"\n✅ VAL  mAP50: {val_metrics.box.map50:.4f} | mAP50-95: {val_metrics.box.map:.4f}")
print(f"✅ TEST mAP50: {test_metrics.box.map50:.4f} | mAP50-95: {test_metrics.box.map:.4f}")

# ==================== 9) EXPORT ONNX ====================
best.export(format="onnx")
print("\n✅ Export ONNX สำเร็จ พร้อมใช้งาน deploy")
print(f"📁 ไฟล์โมเดลอยู่ที่: {results.save_dir}")

✅ พบ GPU: NVIDIA GeForce RTX 4090 -> ใช้ device='0'
🖥️  Device: GPU | Batch: 32

📂 พบไฟล์ภาพทั้งหมด: 2009 ไฟล์
📊 จำนวนกลุ่มทั้งหมด: 99
   เฉลี่ย: 20.29 ไฟล์/กลุ่ม | มากสุด: 25 | น้อยสุด: 1

✅ Split เสร็จ | train=74 groups, val=14 groups, test=11 groups
✅ ไฟล์ที่คัดลอกสำเร็จ: 2009 | ⚠️ ภาพที่ไม่มี label: 0

🔍 ตรวจสอบคุณภาพข้อมูล:
  [train] images=1486 | labels=1486 | class id ผิดพลาด=0
  [val] images=294 | labels=294 | class id ผิดพลาด=0
  [test] images=229 | labels=229 | class id ผิดพลาด=0

✅ สร้างไฟล์ /home/ds/Desktop/Henry-20260923T032836Z-1-001/Henry/Road_Detection/dataset_clean/data.yaml สำเร็จ
✅ โหลด yolo11m.pt จากอินเทอร์เน็ต

🚀 เริ่มเทรนโมเดล...
✅ แปลง polygon labels เป็น bounding boxes แล้ว
Ultralytics 8.4.160 🚀 Python-3.14.6 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 24080MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=9.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosai

[W923 11:41:37.543104455 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1644167168 bytes (free: 220463104, total: 25249579008).
[W923 11:41:37.604496618 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 205520896 bytes (free: 155451392, total: 25249579008).
[W923 11:41:37.604808428 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 205520896 bytes (free: 155451392, total: 25249579008).


train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2275.0±557.0 MB/s, size: 94.2 KB)
train: Scanning /home/ds/Desktop/Henry-20260923T032836Z-1-001/Henry/Road_Detection/dataset_clean/labels/train.cache... 1486 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1486/1486 271.0Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 723.4±161.2 MB/s, size: 89.0 KB)
val: Scanning /home/ds/Desktop/Henry-20260923T032836Z-1-001/Henry/Road_Detection/dataset_clean/labels/val.cache... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 53.6Mit/s 0.0s
: 0% ──────────── 0/47  13.8s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/200      21.9G      3.059      5.268        2.4         58        896: 100% ━━━━━━━━━━━━ 93/93 3.6it/s 26.1s0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.5it/s 2.0s0.3ss
                   all        294        772       0.36     0.0111    0.00